In [18]:
import pickle
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("complete_dataset.csv")

print(df.shape)
df.head()

(195909, 115)


,match_id,source_file,turn,terrain,weather,trick_room,tailwind_p1,tailwind_p2,hazards_p1,hazards_p2,...,active_p1_name_def_stat,active_p1_name_spa_stat,active_p1_name_spd_stat,active_p1_name_spe_stat,active_p2_name_hp_stat,active_p2_name_atk_stat,active_p2_name_def_stat,active_p2_name_spa_stat,active_p2_name_spd_stat,active_p2_name_spe_stat
0,1,gen9ou-2534279795.csv,1,none,none,0,0,0,[],[],...,90.0,130.0,90.0,95.0,75.0,50.0,80.0,95.0,90.0,65.0
1,1,gen9ou-2534279795.csv,1,none,none,0,0,0,[],[],...,90.0,170.0,100.0,95.0,75.0,50.0,80.0,95.0,90.0,65.0
2,1,gen9ou-2534279795.csv,1,none,none,0,0,0,[],[],...,100.0,120.0,90.0,95.0,75.0,50.0,80.0,95.0,90.0,65.0
3,1,gen9ou-2534279795.csv,2,none,none,0,0,0,[],[],...,90.0,130.0,90.0,95.0,75.0,50.0,80.0,95.0,90.0,65.0
4,1,gen9ou-2534279795.csv,2,none,none,0,0,0,[],[],...,90.0,170.0,100.0,95.0,75.0,50.0,80.0,95.0,90.0,65.0


In [3]:
df["action_p1"] = df["choice_p1"].apply(
    lambda x: 1 if isinstance(x, str) and x.startswith("switch") else 0
)

df["action_p1"].value_counts()

action_p1
0    147483
1     48426
Name: count, dtype: int64

In [4]:
df.columns

Index(['match_id', 'source_file', 'turn', 'terrain', 'weather', 'trick_room',
       'tailwind_p1', 'tailwind_p2', 'hazards_p1', 'hazards_p2',
       ...
       'active_p1_name_spa_stat', 'active_p1_name_spd_stat',
       'active_p1_name_spe_stat', 'active_p2_name_hp_stat',
       'active_p2_name_atk_stat', 'active_p2_name_def_stat',
       'active_p2_name_spa_stat', 'active_p2_name_spd_stat',
       'active_p2_name_spe_stat', 'action_p1'],
      dtype='str', length=116)

In [5]:
def create_features(df):
    df["hp_diff"] = df["active_p1_name_hp_stat"] - df["active_p2_name_hp_stat"]
    df["atk_diff"] = df["active_p1_name_atk_stat"] - df["active_p2_name_atk_stat"]
    df["def_diff"] = df["active_p1_name_def_stat"] - df["active_p2_name_def_stat"]
    df["spa_diff"] = df["active_p1_name_spa_stat"] - df["active_p2_name_spa_stat"]
    df["spd_diff"] = df["active_p1_name_spd_stat"] - df["active_p2_name_spd_stat"]
    df["spe_diff"] = df["active_p1_name_spe_stat"] - df["active_p2_name_spe_stat"]

    df["atk_vs_def"] = df["active_p1_name_atk_stat"] - df["active_p2_name_def_stat"]
    df["spa_vs_spd"] = df["active_p1_name_spa_stat"] - df["active_p2_name_spd_stat"]

    df["speed_diff"] = df["active_p1_name_spe_stat"] - df["active_p2_name_spe_stat"]
    df["speed_advantage"] = (df["speed_diff"] > 0).astype(int)

    df["total_stat_p1"] = (
        df["active_p1_name_hp_stat"] +
        df["active_p1_name_atk_stat"] +
        df["active_p1_name_def_stat"] +
        df["active_p1_name_spa_stat"] +
        df["active_p1_name_spd_stat"] +
        df["active_p1_name_spe_stat"]
    )

    df["total_stat_p2"] = (
        df["active_p2_name_hp_stat"] +
        df["active_p2_name_atk_stat"] +
        df["active_p2_name_def_stat"] +
        df["active_p2_name_spa_stat"] +
        df["active_p2_name_spd_stat"] +
        df["active_p2_name_spe_stat"]
    )

    df["total_diff"] = df["total_stat_p1"] - df["total_stat_p2"]

    return df

df = create_features(df)

In [21]:
df.to_csv("complete_dataset_for_processing.csv", index=False)

print("Dataset sauvegardé : complete_dataset_for_processing.csv")

Dataset sauvegardé : complete_dataset_for_processing.csv


In [6]:
match_ids = df["match_id"].unique()

print("Total matches:", len(match_ids))

Total matches: 4868


In [7]:
train_matches, test_matches = train_test_split(
    match_ids,
    test_size=0.3,
    random_state=42
)

In [8]:
train_df = df[df["match_id"].isin(train_matches)]
test_df = df[df["match_id"].isin(test_matches)]

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train matches:", train_df["match_id"].nunique())
print("Test matches:", test_df["match_id"].nunique())

Train rows: 135697
Test rows: 60212
Train matches: 3407
Test matches: 1461


In [9]:
X_train = train_df.drop(columns=["action_p1"])
y_train = train_df["action_p1"]

X_test = test_df.drop(columns=["action_p1"])
y_test = test_df["action_p1"]

In [10]:
cols_to_drop = ["source_file", "choice_p1", "match_id"]

X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

In [11]:
move_cols = [c for c in X_train.columns if "move" in c]

for col in move_cols:
    X_train[col] = X_train[col].fillna("none")
    X_test[col] = X_test[col].fillna("none")

In [12]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, accuracy_score

categorical_cols = X_train.select_dtypes(include="object").columns
numeric_cols = X_train.select_dtypes(exclude="object").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=64,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

print("Training LightGBM...")
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]

for t in [0.3, 0.4, 0.5, 0.6]:
    preds = (probs > t).astype(int)

    report = classification_report(y_test, preds, output_dict=True)

    print(f"\nThreshold {t}")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Recall switch:", report["1"]["recall"])
    print("Precision switch:", report["1"]["precision"])
    print("F1 switch:", report["1"]["f1-score"])

Training LightGBM...


/var/folders/5l/smv47lvj0k5c9wgqbbqqtz940000gn/T/ipykernel_2071/339962640.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include="object").columns


[LightGBM] [Info] Number of positive: 33183, number of negative: 102514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.410397 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18252
[LightGBM] [Info] Number of data points in the train set: 135697, number of used features: 6603
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/silarbinunzio/Documents/GitHub/Pokestral-V2/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Threshold 0.3
Accuracy: 0.6218693948050222
Recall switch: 0.9334120579938332
Precision switch: 0.39543092187543427
F1 switch: 0.5555208496017492

Threshold 0.4
Accuracy: 0.6927190593237228
Recall switch: 0.8754838286426556
Precision switch: 0.4455908377575211
F1 switch: 0.5905912550893964

Threshold 0.5
Accuracy: 0.752574237693483
Recall switch: 0.7740602243652824
Precision switch: 0.5074183976261127
F1 switch: 0.612998753117207

Threshold 0.6
Accuracy: 0.7925662658606258
Recall switch: 0.6303221150692121
Precision switch: 0.5836117354066696
F1 switch: 0.606068252065855


In [13]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

probs = model.predict_proba(X_test)[:, 1]

threshold = 0.6
preds = (probs > threshold).astype(int)

print("Accuracy:", accuracy_score(y_test, preds))

print("\nClassification Report:")
print(classification_report(y_test, preds))

cm = confusion_matrix(y_test, preds)

print("\nConfusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nDétail :")
print(f"True Negatives (stay bien prédit): {tn}")
print(f"False Positives (switch inutile): {fp}")
print(f"False Negatives (switch raté): {fn}")
print(f"True Positives (switch correct): {tp}")

/Users/silarbinunzio/Documents/GitHub/Pokestral-V2/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.7925662658606258

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.85      0.86     44969
           1       0.58      0.63      0.61     15243

    accuracy                           0.79     60212
   macro avg       0.73      0.74      0.73     60212
weighted avg       0.80      0.79      0.80     60212


Confusion Matrix:
[[38114  6855]
 [ 5635  9608]]

Détail :
True Negatives (stay bien prédit): 38114
False Positives (switch inutile): 6855
False Negatives (switch raté): 5635
True Positives (switch correct): 9608


In [16]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LGBMClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ))
    ]
)

param_dist = {
    "classifier__n_estimators": randint(100, 300),
    "classifier__learning_rate": uniform(0.03, 0.07),
    "classifier__num_leaves": randint(20, 80),
    "classifier__max_depth": randint(5, 20),
}

random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=8,
    scoring="f1",
    cv=2,
    verbose=2,
    n_jobs=1,
    random_state=42
)

print("Training RandomizedSearch (safe mode)...")
random_search.fit(X_train, y_train)

print("\nBest params:")
print(random_search.best_params_)

Training RandomizedSearch (safe mode)...
Fitting 2 folds for each of 8 candidates, totalling 16 fits
[LightGBM] [Info] Number of positive: 16591, number of negative: 51257
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.672914 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15703
[LightGBM] [Info] Number of data points in the train set: 67848, number of used features: 5395
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


/Users/silarbinunzio/Documents/GitHub/Pokestral-V2/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[CV] END classifier__learning_rate=0.056217808319315374, classifier__max_depth=17, classifier__n_estimators=114, classifier__num_leaves=62; total time=  47.8s
[LightGBM] [Info] Number of positive: 16592, number of negative: 51257
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.172546 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15547
[LightGBM] [Info] Number of data points in the train set: 67849, number of used features: 5319
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/silarbinunzio/Documents/GitHub/Pokestral-V2/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[CV] END classifier__learning_rate=0.056217808319315374, classifier__max_depth=17, classifier__n_estimators=114, classifier__num_leaves=62; total time=  38.5s
[LightGBM] [Info] Number of positive: 16591, number of negative: 51257
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.756652 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15703
[LightGBM] [Info] Number of data points in the train set: 67848, number of used features: 5395
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

...

[CV] END classifier__learning_rate=0.051296957007167646, classifier__max_depth=10, classifier__n_estimators=188, classifier__num_leaves=68; total time=  38.5s
[LightGBM] [Info] Number of positive: 16591, number of negative: 51257
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.351514 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15703
[LightGBM] [Info] Number of data points in the train set: 67848, number of used features: 5395
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


/Users/silarbinunzio/Documents/GitHub/Pokestral-V2/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[CV] END classifier__learning_rate=0.06673422621808725, classifier__max_depth=19, classifier__n_estimators=269, classifier__num_leaves=47; total time=  42.6s
[LightGBM] [Info] Number of positive: 16592, number of negative: 51257
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.042665 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15547
[LightGBM] [Info] Number of data points in the train set: 67849, number of used features: 5319
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/silarbinunzio/Documents/GitHub/Pokestral-V2/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[CV] END classifier__learning_rate=0.06673422621808725, classifier__max_depth=19, classifier__n_estimators=269, classifier__num_leaves=47; total time=  44.4s
[LightGBM] [Info] Number of positive: 33183, number of negative: 102514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 3.570963 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18252
[LightGBM] [Info] Number of data points in the train set: 135697, number of used features: 6603
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

Best params:
{'classifier__learning_rate': np.float64(0.06673422621808725), 'classifier__max_depth': 19, 'classifier__n_estimators': 269, 'classifier__num_leaves': 47}


In [17]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

best_params = {
    "n_estimators": 269,
    "learning_rate": 0.06673422621808725,
    "max_depth": 19,
    "num_leaves": 47
}

best_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LGBMClassifier(
            **best_params,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

print("Training best LightGBM...")
best_model.fit(X_train, y_train)

probs = best_model.predict_proba(X_test)[:, 1]

for t in [0.4, 0.5, 0.6]:
    preds = (probs > t).astype(int)

    print(f"\nThreshold {t}")
    print("Accuracy:", accuracy_score(y_test, preds))
    print(classification_report(y_test, preds))

Training best LightGBM...
[LightGBM] [Info] Number of positive: 33183, number of negative: 102514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.883490 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18252
[LightGBM] [Info] Number of data points in the train set: 135697, number of used features: 6603
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/silarbinunzio/Documents/GitHub/Pokestral-V2/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Threshold 0.4
Accuracy: 0.6845645386301734
              precision    recall  f1-score   support

           0       0.94      0.62      0.75     44969
           1       0.44      0.88      0.58     15243

    accuracy                           0.68     60212
   macro avg       0.69      0.75      0.67     60212
weighted avg       0.81      0.68      0.70     60212


Threshold 0.5
Accuracy: 0.7490699528333222
              precision    recall  f1-score   support

           0       0.91      0.74      0.81     44969
           1       0.50      0.78      0.61     15243

    accuracy                           0.75     60212
   macro avg       0.71      0.76      0.71     60212
weighted avg       0.81      0.75      0.76     60212


Threshold 0.6
Accuracy: 0.7915531787683519
              precision    recall  f1-score   support

           0       0.87      0.84      0.86     44969
           1       0.58      0.64      0.61     15243

    accuracy                           0.79     60

In [19]:
class ModelWithThreshold:
    def __init__(self, model, threshold=0.6):
        self.model = model
        self.threshold = threshold

    def predict(self, X):
        probs = self.model.predict_proba(X)[:, 1]
        return (probs > self.threshold).astype(int)

    def predict_proba(self, X):
        return self.model.predict_proba(X)

final_model = ModelWithThreshold(best_model, threshold=0.6)

with open("pokemon_switch_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

print("Modèle sauvegardé : pokemon_switch_model.pkl")

Modèle sauvegardé : pokemon_switch_model.pkl
